# Extract SOS validation results — Unity

This notebook reads every SOS NetCDF file, extracts `/validation/moi` and `/validation/flpe`, counts valid results, and exports portable CSV files. A result is counted when `has_validation == 1` and `nbias` is finite.

Requirements: `numpy`, `pandas`, and `netCDF4`. Run all cells, then download the complete `unity_validation_export` directory. The comparison notebook needs that directory unchanged.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import gc
import json
import platform
import re
import sys

import numpy as np
import pandas as pd
from netCDF4 import Dataset, chartostring

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)

In [ ]:
# Only this cell normally needs editing.
RUN_LABEL = 'Unity'
SOS_DIR = Path('/nas/cee-water/cjgleason/Yushan/Confluence_Aug/confluence_global_v17c_gagecorr/global_v17c_gagecorr_mnt/output/sos')
EXPORT_DIR = Path.cwd() / 'unity_validation_export'

# Search all NetCDF files directly in SOS_DIR. Set RECURSIVE=True if needed.
FILE_GLOB = "*.nc"
RECURSIVE = False
VALIDATION_GROUPS = ("moi", "flpe")

# Only validation rows are read. This limit bounds peak memory for unusually
# large validation sets; 20,000 rows is conservative on shared HPC notebooks.
VALIDATION_ROW_CHUNK_SIZE = 20_000

# nBias is sufficient for the requested count/precision comparison. Set this
# to None to export every aligned numeric metric after the nBias run succeeds.
METRICS_TO_EXPORT = ("nbias",)
EXPORT_FLOAT_HEX = True

# With STRICT=True, extraction errors are reported after all readable CSVs are saved.
STRICT = True

print("SOS_DIR:", SOS_DIR)
print("EXPORT_DIR:", EXPORT_DIR.resolve())

## Extraction helpers

The extractor first selects `has_validation == 1`, then reads those rows in bounded chunks and streams them directly to partial CSV files. By default only `nbias` is exported. Each value also has a float64 hexadecimal representation.

In [ ]:
def clean_text(value):
    """Decode one bytes/string scalar and remove NetCDF padding."""
    if isinstance(value, (bytes, np.bytes_)):
        value = bytes(value).decode("utf-8", errors="replace")
    return str(value).replace("\x00", "").strip()


def decode_text_data(raw):
    """Decode a NetCDF char/VLEN selection into a text ndarray."""
    values = np.asarray(np.ma.filled(raw, b""))
    if values.dtype.kind == "S" and values.dtype.itemsize == 1 and values.ndim:
        values = np.asarray(chartostring(values, encoding="utf-8"))
    if values.ndim == 0:
        return np.asarray(clean_text(values.item()), dtype=object)
    return np.vectorize(clean_text, otypes=[object])(values)


def selected_data(variable, row_indices=None):
    if row_indices is None:
        return variable[:]
    return variable[np.asarray(row_indices, dtype=np.int64), ...]


def read_text_selection(variable, row_indices=None):
    return decode_text_data(selected_data(variable, row_indices))


def numeric_data_to_float64(variable, raw):
    masked = np.ma.asarray(raw).astype(np.float64)
    values = np.asarray(np.ma.filled(masked, np.nan), dtype=np.float64)
    fill_value = getattr(variable, "_FillValue", None)
    if fill_value is not None:
        try:
            values[np.isclose(values, float(fill_value), equal_nan=False)] = np.nan
        except (TypeError, ValueError):
            pass
    valid_min = getattr(variable, "valid_min", None)
    valid_max = getattr(variable, "valid_max", None)
    valid_range = getattr(variable, "valid_range", None)
    if valid_range is not None and np.asarray(valid_range).size == 2:
        valid_min, valid_max = np.asarray(valid_range).reshape(-1)[:2]
    if valid_min is not None:
        values[values < float(valid_min)] = np.nan
    if valid_max is not None:
        values[values > float(valid_max)] = np.nan
    return values


def read_numeric_selection(variable, row_indices=None):
    return numeric_data_to_float64(variable, selected_data(variable, row_indices))


def is_numeric_variable(variable):
    try:
        return np.issubdtype(np.dtype(variable.dtype), np.number)
    except TypeError:
        return False


def normalize_algorithm(value):
    text = clean_text(value).lower().replace("-", "_").replace(" ", "_")
    text = re.sub(r"_+", "_", text)
    text = re.sub(r"^(flpe|moi)_+", "", text)
    text = re.sub(r"_+(flpe|moi)$", "", text)
    return text.strip("_")


def normalize_reach_id(value):
    if value is None or np.ma.is_masked(value):
        return ""
    text = clean_text(value)
    if not text or text.lower() in {"nan", "none", "<na>"}:
        return ""
    try:
        number = float(text)
    except (TypeError, ValueError):
        return text
    if not np.isfinite(number) or number <= 0:
        return ""
    return str(int(round(number)))


def normalize_gage_id(value):
    return re.sub(r"[^a-z0-9]+", "", clean_text(value).lower())


def partition_key(path):
    match = re.match(r"([a-z]{2})_sword_", path.name, flags=re.IGNORECASE)
    if match:
        return match.group(1).upper()
    prefix = re.split(r"_SOS_results_", path.stem, maxsplit=1, flags=re.IGNORECASE)[0]
    return re.sub(r"[^a-z0-9]+", "_", prefix.lower()).strip("_")


def candidate_reach_variables(dataset):
    candidates = []
    if "reaches" in dataset.groups:
        group = dataset.groups["reaches"]
        for name in ("reach_id", "reach_ids", "reachid"):
            if name in group.variables:
                candidates.append((f"/reaches/{name}", group.variables[name]))
    for name in ("reach_id", "reach_ids", "reachid"):
        if name in dataset.variables:
            candidates.append((f"/{name}", dataset.variables[name]))
    return candidates


def find_selected_reach_ids(dataset, n_reaches, row_indices):
    for source, variable in candidate_reach_variables(dataset):
        if not variable.shape or variable.shape[0] != n_reaches:
            continue
        try:
            if is_numeric_variable(variable):
                values = np.ma.filled(selected_data(variable, row_indices), np.nan)
            else:
                values = read_text_selection(variable, row_indices)
            values = np.asarray(values, dtype=object).reshape(-1)
        except Exception:
            continue
        if values.size == len(row_indices):
            return values, source
    fallback = np.asarray([f"row:{index}" for index in row_indices], dtype=object)
    return fallback, "fallback_row_index"


def align_selected_algorithms(variable, n_reaches, n_algorithms, row_indices):
    if variable.shape and variable.shape[0] == n_reaches:
        values = read_text_selection(variable, row_indices)
    else:
        values = read_text_selection(variable)
    values = np.asarray(values, dtype=object)
    target = (len(row_indices), n_algorithms)
    if values.shape == target:
        return values
    if values.ndim == 1 and values.size == n_algorithms:
        return np.broadcast_to(values.reshape(1, -1), target).copy()
    if values.size == target[0] * target[1]:
        return values.reshape(target)
    raise ValueError(f"algo_names shape {values.shape} cannot align to {target}")


def read_selected_gageids(group, n_reaches, row_indices):
    if "gageid" not in group.variables:
        return np.full(len(row_indices), "", dtype=object)
    variable = group.variables["gageid"]
    if not variable.shape or variable.shape[0] != n_reaches:
        raise ValueError(f"gageid first dimension does not match {n_reaches} reach rows")
    values = np.asarray(read_text_selection(variable, row_indices), dtype=object).reshape(-1)
    if values.size != len(row_indices):
        raise ValueError(f"gageid selection has {values.size} values for {len(row_indices)} rows")
    return values


def aligned_numeric_metric(variable, n_reaches, n_algorithms):
    return (
        variable.shape == (n_reaches, n_algorithms)
        or (n_algorithms == 1 and variable.shape == (n_reaches,))
    )


def inspect_validation_group(dataset, group_name, file_metadata):
    validation = dataset.groups["validation"]
    if group_name not in validation.groups:
        raise KeyError(f"/validation/{group_name} is missing")
    group = validation.groups[group_name]
    for required in ("nbias", "algo_names", "has_validation"):
        if required not in group.variables:
            raise KeyError(f"/validation/{group_name}/{required} is missing")

    nbias_variable = group.variables["nbias"]
    if len(nbias_variable.shape) == 1:
        n_reaches, n_algorithms = nbias_variable.shape[0], 1
    elif len(nbias_variable.shape) == 2:
        n_reaches, n_algorithms = nbias_variable.shape
    else:
        raise ValueError(f"nbias must be 1-D or 2-D, found {nbias_variable.shape}")

    has_validation = read_numeric_selection(group.variables["has_validation"]).reshape(-1)
    if has_validation.size != n_reaches:
        raise ValueError(
            f"has_validation length {has_validation.size} != reach rows {n_reaches}"
        )
    validation_indices = np.flatnonzero(np.isfinite(has_validation) & (has_validation == 1))

    available_metrics = []
    variable_rows = []
    requested = None if METRICS_TO_EXPORT is None else set(METRICS_TO_EXPORT)
    for variable_name, variable in group.variables.items():
        aligned = (
            variable_name != "has_validation"
            and is_numeric_variable(variable)
            and aligned_numeric_metric(variable, n_reaches, n_algorithms)
        )
        should_export = aligned and (
            requested is None or variable_name in requested or variable_name == "nbias"
        )
        if should_export:
            available_metrics.append(variable_name)
        variable_rows.append(
            {
                **file_metadata,
                "group": group_name,
                "variable": variable_name,
                "dtype": str(variable.dtype),
                "dimensions": "|".join(variable.dimensions),
                "shape": "x".join(str(value) for value in variable.shape),
                "aligned_result_metric": aligned,
                "exported_metric": should_export,
            }
        )
    if "nbias" not in available_metrics:
        raise ValueError("nbias could not be aligned/exported")

    return {
        "group": group,
        "n_reaches": n_reaches,
        "n_algorithms": n_algorithms,
        "validation_indices": validation_indices,
        "metrics": available_metrics,
        "variable_rows": variable_rows,
    }


def extract_validation_chunk(dataset, info, row_indices, group_name, file_metadata):
    group = info["group"]
    n_reaches = info["n_reaches"]
    n_algorithms = info["n_algorithms"]
    row_indices = np.asarray(row_indices, dtype=np.int64)
    selected_count = len(row_indices)
    target_shape = (selected_count, n_algorithms)

    algorithms = align_selected_algorithms(
        group.variables["algo_names"], n_reaches, n_algorithms, row_indices
    )
    algorithms_raw = np.asarray(algorithms, dtype=object).reshape(-1)
    algorithms_normalized = np.asarray(
        [normalize_algorithm(value) for value in algorithms_raw], dtype=object
    )
    blank = algorithms_normalized == ""
    if blank.any():
        algorithm_columns = np.tile(np.arange(n_algorithms), selected_count)
        algorithms_normalized[blank] = [
            f"algorithm_column_{index}" for index in algorithm_columns[blank]
        ]

    gageids = read_selected_gageids(group, n_reaches, row_indices)
    reach_ids_raw, reach_id_source = find_selected_reach_ids(
        dataset, n_reaches, row_indices
    )
    reach_ids = np.asarray([normalize_reach_id(value) for value in reach_ids_raw], dtype=object)
    gageid_keys = np.asarray([normalize_gage_id(value) for value in gageids], dtype=object)

    metric_arrays = {}
    for metric_name in info["metrics"]:
        values = read_numeric_selection(group.variables[metric_name], row_indices)
        if n_algorithms == 1 and values.shape == (selected_count,):
            values = values.reshape(-1, 1)
        if values.shape != target_shape:
            raise ValueError(
                f"{metric_name} selection shape {values.shape} != {target_shape}"
            )
        metric_arrays[metric_name] = values

    reach_index_flat = np.repeat(row_indices, n_algorithms)
    algorithm_index_flat = np.tile(np.arange(n_algorithms), selected_count)
    nbias_flat = metric_arrays["nbias"].reshape(-1)
    local_ids = np.asarray(
        [
            f"{file_metadata['relative_path']}|{group_name}|{reach}|{algorithm}"
            for reach, algorithm in zip(reach_index_flat, algorithm_index_flat)
        ],
        dtype=object,
    )
    cells = pd.DataFrame(
        {
            **{name: value for name, value in file_metadata.items()},
            "local_cell_id": local_ids,
            "group": group_name,
            "row_index": reach_index_flat,
            "algorithm_index": algorithm_index_flat,
            "reach_id_raw": np.repeat(
                np.asarray([clean_text(value) for value in reach_ids_raw], dtype=object),
                n_algorithms,
            ),
            "reach_id": np.repeat(reach_ids, n_algorithms),
            "reach_id_source": reach_id_source,
            "gageid": np.repeat(
                np.asarray([clean_text(value) for value in gageids], dtype=object),
                n_algorithms,
            ),
            "gageid_key": np.repeat(gageid_keys, n_algorithms),
            "algorithm_raw": np.asarray([clean_text(value) for value in algorithms_raw]),
            "algorithm": algorithms_normalized,
            "has_validation": 1,
            "nbias_is_finite": np.isfinite(nbias_flat),
            "result_present": np.isfinite(nbias_flat),
        }
    )

    value_frames = []
    for metric_name, matrix in metric_arrays.items():
        flat = matrix.reshape(-1)
        if EXPORT_FLOAT_HEX:
            hex_values = np.asarray([float(np.float64(value)).hex() for value in flat])
        else:
            hex_values = np.full(flat.size, "", dtype=object)
        value_frames.append(
            pd.DataFrame(
                {
                    **{name: value for name, value in file_metadata.items()},
                    "local_cell_id": local_ids,
                    "group": group_name,
                    "row_index": reach_index_flat,
                    "algorithm_index": algorithm_index_flat,
                    "reach_id": np.repeat(reach_ids, n_algorithms),
                    "gageid_key": np.repeat(gageid_keys, n_algorithms),
                    "algorithm": algorithms_normalized,
                    "has_validation": 1,
                    "metric": metric_name,
                    "metric_dtype": str(group.variables[metric_name].dtype),
                    "value": flat,
                    "value_hex64": hex_values,
                    "is_finite": np.isfinite(flat),
                }
            )
        )
    values = pd.concat(value_frames, ignore_index=True)
    return cells, values


def attach_stable_keys_to_chunk(cells, values, occurrence_state):
    occurrences = []
    record_keys = []
    key_columns = ["group", "partition_key", "reach_id", "gageid_key", "algorithm"]
    for row in cells[key_columns].itertuples(index=False, name=None):
        occurrence = occurrence_state[row]
        occurrence_state[row] += 1
        occurrences.append(occurrence)
        record_keys.append("|".join([*(str(value) for value in row), str(occurrence)]))
    cells = cells.copy()
    cells["occurrence"] = occurrences
    cells["record_key"] = record_keys
    lookup = cells[["local_cell_id", "occurrence", "record_key"]]
    values = values.merge(lookup, on="local_cell_id", how="left", validate="many_to_one")
    return cells, values


def update_count_accumulator(accumulator, cells):
    group_name = cells["group"].iloc[0]
    for algorithm, frame in cells.groupby("algorithm", sort=False):
        for key in ((group_name, algorithm), (group_name, "ALL")):
            entry = accumulator.setdefault(
                key,
                {
                    "total_cells": 0,
                    "validation_flag_count": 0,
                    "finite_nbias_count": 0,
                    "result_count": 0,
                    "reaches": set(),
                    "gages": set(),
                },
            )
            result_frame = frame.loc[frame["result_present"]]
            entry["total_cells"] += len(frame)
            entry["validation_flag_count"] += len(frame)
            entry["finite_nbias_count"] += int(frame["nbias_is_finite"].sum())
            entry["result_count"] += int(frame["result_present"].sum())
            entry["reaches"].update(result_frame["reach_id"].astype(str))
            entry["gages"].update(
                value for value in result_frame["gageid_key"].astype(str) if value
            )


def count_accumulator_frame(accumulator):
    rows = []
    for (group_name, algorithm), entry in sorted(accumulator.items()):
        rows.append(
            {
                "run_label": RUN_LABEL,
                "group": group_name,
                "algorithm": algorithm,
                "total_cells": entry["total_cells"],
                "validation_flag_count": entry["validation_flag_count"],
                "finite_nbias_count": entry["finite_nbias_count"],
                "result_count": entry["result_count"],
                "unique_result_reaches": len(entry["reaches"]),
                "unique_result_gages": len(entry["gages"]),
            }
        )
    return pd.DataFrame(rows)


class StreamingCsvWriter:
    """Append chunks to a partial CSV and publish only after extraction."""
    def __init__(self, final_path):
        self.final_path = Path(final_path)
        self.partial_path = self.final_path.with_name(self.final_path.name + ".partial")
        self.started = False
        self.rows = 0

    def append(self, frame):
        if frame.empty:
            return
        frame.to_csv(
            self.partial_path,
            mode="a" if self.started else "w",
            header=not self.started,
            index=False,
            float_format="%.17g",
            na_rep="NaN",
        )
        self.started = True
        self.rows += len(frame)

    def publish(self):
        if not self.started:
            raise RuntimeError(f"No rows were written to {self.final_path.name}")
        self.partial_path.replace(self.final_path)


def atomic_to_csv(frame, final_path):
    final_path = Path(final_path)
    partial_path = final_path.with_name(final_path.name + ".partial")
    frame.to_csv(
        partial_path, index=False, float_format="%.17g", na_rep="NaN"
    )
    partial_path.replace(final_path)


def safe_json_value(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    return value

## Run extraction and write CSV files

In [ ]:
if not SOS_DIR.is_dir():
    raise NotADirectoryError(f"SOS_DIR does not exist: {SOS_DIR}")

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
paths = sorted(SOS_DIR.rglob(FILE_GLOB) if RECURSIVE else SOS_DIR.glob(FILE_GLOB))
if not paths:
    raise FileNotFoundError(f"No files matching {FILE_GLOB!r} under {SOS_DIR}")

cells_writer = StreamingCsvWriter(EXPORT_DIR / "validation_cells.csv")
values_writer = StreamingCsvWriter(EXPORT_DIR / "validation_values.csv")
occurrence_state = defaultdict(int)
count_accumulator = {}
file_count_accumulator = defaultdict(
    lambda: {"validation_flag_count": 0, "result_count": 0, "total_cells": 0}
)
inventory_rows = []
issue_rows = []
variable_rows = []
fallback_rows = 0

print(f"Found {len(paths):,} candidate NetCDF files")
for file_number, path in enumerate(paths, start=1):
    relative_path = str(path.relative_to(SOS_DIR))
    metadata = {
        "run_label": RUN_LABEL,
        "source_file": path.name,
        "relative_path": relative_path,
        "partition_key": partition_key(path),
        "file_size_bytes": path.stat().st_size,
    }
    inventory = {**metadata, "extract_status": "pending", "validation_groups": ""}
    print(f"\n[{file_number}/{len(paths)}] {relative_path}")
    try:
        with Dataset(path, mode="r") as dataset:
            if "validation" not in dataset.groups:
                inventory["extract_status"] = "skipped_no_validation_group"
                inventory_rows.append(inventory)
                continue
            available = sorted(dataset.groups["validation"].groups)
            inventory["validation_groups"] = "|".join(available)
            extracted_groups = []
            for group_name in VALIDATION_GROUPS:
                try:
                    info = inspect_validation_group(dataset, group_name, metadata)
                    variable_rows.extend(info["variable_rows"])
                    validation_indices = info["validation_indices"]
                    print(
                        f"  {group_name}: {info['n_reaches']:,} reach rows, "
                        f"{info['n_algorithms']} algorithms, "
                        f"{len(validation_indices):,} validation rows, "
                        f"metrics={info['metrics']}"
                    )
                    for chunk_start in range(
                        0, len(validation_indices), VALIDATION_ROW_CHUNK_SIZE
                    ):
                        row_indices = validation_indices[
                            chunk_start : chunk_start + VALIDATION_ROW_CHUNK_SIZE
                        ]
                        cells, values = extract_validation_chunk(
                            dataset, info, row_indices, group_name, metadata
                        )
                        cells, values = attach_stable_keys_to_chunk(
                            cells, values, occurrence_state
                        )
                        update_count_accumulator(count_accumulator, cells)
                        for algorithm, frame in cells.groupby("algorithm", sort=False):
                            key = (
                                relative_path,
                                metadata["partition_key"],
                                group_name,
                                algorithm,
                            )
                            file_entry = file_count_accumulator[key]
                            file_entry["validation_flag_count"] += len(frame)
                            file_entry["result_count"] += int(
                                frame["result_present"].sum()
                            )
                            file_entry["total_cells"] += len(frame)
                        fallback_rows += int(
                            (cells["reach_id_source"] == "fallback_row_index").sum()
                        )
                        cells_writer.append(cells)
                        values_writer.append(values)
                        del cells, values
                    extracted_groups.append(group_name)
                    del info
                    gc.collect()
                except Exception as exc:
                    issue_rows.append(
                        {
                            **metadata,
                            "severity": "error",
                            "group": group_name,
                            "message": f"{type(exc).__name__}: {exc}",
                        }
                    )
                    print(f"  ERROR {group_name}: {type(exc).__name__}: {exc}")
            inventory["extract_status"] = (
                "ok:" + "|".join(extracted_groups)
                if extracted_groups
                else "error_no_groups_extracted"
            )
    except Exception as exc:
        inventory["extract_status"] = "error_opening_file"
        issue_rows.append(
            {
                **metadata,
                "severity": "error",
                "group": "",
                "message": f"{type(exc).__name__}: {exc}",
            }
        )
        print(f"  ERROR opening file: {type(exc).__name__}: {exc}")
    inventory_rows.append(inventory)
    gc.collect()

issue_columns = [
    "run_label", "source_file", "relative_path", "partition_key",
    "file_size_bytes", "severity", "group", "message",
]
inventory_df = pd.DataFrame(inventory_rows)
issues_df = pd.DataFrame(issue_rows, columns=issue_columns)
variables_df = pd.DataFrame(variable_rows)
counts_df = count_accumulator_frame(count_accumulator)

atomic_to_csv(inventory_df, EXPORT_DIR / "file_inventory.csv")
atomic_to_csv(issues_df, EXPORT_DIR / "extraction_issues.csv")
atomic_to_csv(variables_df, EXPORT_DIR / "variable_inventory.csv")

if not cells_writer.started or not values_writer.started:
    raise RuntimeError(
        "No validation records were extracted. Inspect file_inventory.csv and "
        "extraction_issues.csv."
    )

cells_writer.publish()
values_writer.publish()
atomic_to_csv(counts_df, EXPORT_DIR / "count_summary.csv")

file_count_rows = []
for key, entry in sorted(file_count_accumulator.items()):
    relative_path, current_partition, group_name, algorithm = key
    file_count_rows.append(
        {
            "relative_path": relative_path,
            "partition_key": current_partition,
            "group": group_name,
            "algorithm": algorithm,
            **entry,
        }
    )
atomic_to_csv(pd.DataFrame(file_count_rows), EXPORT_DIR / "count_by_file.csv")

run_info = {
    "run_label": RUN_LABEL,
    "sos_dir": str(SOS_DIR),
    "export_dir": str(EXPORT_DIR.resolve()),
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "file_glob": FILE_GLOB,
    "recursive": RECURSIVE,
    "validation_row_chunk_size": VALIDATION_ROW_CHUNK_SIZE,
    "metrics_to_export": METRICS_TO_EXPORT,
    "candidate_file_count": len(paths),
    "extracted_cell_count": cells_writer.rows,
    "extracted_value_count": values_writer.rows,
    "issue_count": len(issues_df),
    "count_definition": "result_count = has_validation == 1 AND finite nbias",
}
run_info_path = EXPORT_DIR / "run_info.json"
run_info_partial = run_info_path.with_name(run_info_path.name + ".partial")
with run_info_partial.open("w", encoding="utf-8") as stream:
    json.dump(
        run_info, stream, ensure_ascii=False, indent=2, default=safe_json_value
    )
run_info_partial.replace(run_info_path)

print(f"\nExport complete: {EXPORT_DIR.resolve()}")
print("Only has_validation == 1 rows were materialized.")
print("Count definition: result_count = has_validation == 1 AND finite nbias")
display(counts_df.sort_values(["group", "algorithm"]).reset_index(drop=True))
print(f"Rows using fallback row index instead of reach ID: {fallback_rows:,}")
print(f"Extraction issues: {len(issues_df):,}")
if not issues_df.empty:
    display(issues_df)
    if STRICT:
        raise RuntimeError(
            "Extraction finished with errors. Completed CSV files were saved; inspect "
            "extraction_issues.csv. Set STRICT=False only if the affected files/groups "
            "are intentionally out of scope."
        )

## Files to download

Download the entire export directory. Important files are `count_summary.csv`, `validation_cells.csv`, `validation_values.csv`, `file_inventory.csv`, and `extraction_issues.csv`.